In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import matplotlib.style as style
from statsmodels.tsa.api import seasonal_decompose
import datetime
import time
import pandas.util.testing as tm
import matplotlib.pyplot as mp
style.use('ggplot')

# **Inputs**

In [ ]:
# INPUTS

data = datetime.date(2023, 11, 22)
sugestao_anterior = "df_sugestao_nov23_late.csv"
hora = '1520'
data2 = data.replace(day=1)
data_penultimo_fim = data2 - datetime.timedelta(days=1)
data_antepenultimo_inicio = data_penultimo_fim.replace(day=1)
data_antepenultimo_fim = data_antepenultimo_inicio - datetime.timedelta(days=1)
#hora = '0830'
#data_inicio = data.replace(day=1)

#SET UP
dia = data.day
mes = data.month
mes_anterior = data_penultimo_fim.month
ano = data.year
s_dia = '{:02d}'.format(data.day)
s_mes = '{:02d}'.format(data.month)
s_mes_penultimo = '{:02d}'.format(data_penultimo_fim.month)
s_dia_penultimo_fim = '{:02d}'.format(data_penultimo_fim.day)
s_ano = str(ano)
s_data = s_ano+ s_mes + s_dia
s_data_penultimo_fim = s_ano + s_mes_penultimo + s_dia_penultimo_fim
dia_mes = dia

In [ ]:
# Reloading
with open(file = "df_"+ s_data_penultimo_fim +"_tot_svm1MABC.csv", mode = 'r') as f:
    df_svmMABC = pd.read_csv(f, index_col=0)

with open(file = "df_"+ s_data_penultimo_fim +"_tot_svm2MABC.csv", mode = 'r') as f:
    df_svm2MABC = pd.read_csv(f, index_col=0)

with open(file = "df_"+ s_data_penultimo_fim +"_tot_svm3MABC.csv", mode = 'r') as f:
    df_svm3MABC = pd.read_csv(f, index_col=0)

with open(file = "df_"+ s_data_penultimo_fim +"_tot_linearMABC.csv", mode = 'r') as f:
    df_linear = pd.read_csv(f, index_col=0)

with open(file = "df_hist_"+ s_data_penultimo_fim + "_tot1MABC.csv", mode = 'r') as f:
    df_hist_nov_totMABC = pd.read_csv(f, index_col=0)

#Na parcial
with open(file = "df_estoque_ABC_"+ s_data + "_raw.csv", mode = 'r') as f:
    df_estoque = pd.read_csv(f, index_col=0)
#Na parcial
with open(file = sugestao_anterior, mode = 'r') as f:
    df_ultima_sugestao_entradaABC = pd.read_csv(f, index_col=0)
#Na parcial
with open(file = "df_hist_"+ s_data + "_linha.csv", mode = 'r') as f:
    df_vendas_atualizadas_entrada = pd.read_csv(f, index_col=0)
#Na parcial
with open(file = "pedidos_em_aberto_"+ s_data + hora + ".csv", mode = 'r') as f:
    df_pedidos = pd.read_csv(f, index_col=0)





with open(file = "CPP_latenov_curvaA.csv", mode = 'r') as f:
    df_ultima_sugestao_entrada = pd.read_csv(f, index_col=0)

with open(file = "df_aleatorio_20221214.csv", mode = 'r') as f:
    df_aleatorio = pd.read_csv(f, index_col=0)


with open(file = "df_coberturas_fornecedores.csv", mode = 'r') as f:
    df_coberturas = pd.read_csv(f, index_col=0)





In [ ]:
def normalizando(dataframe):
    df = dataframe
    tamanho = len(df)
    final = pd.json_normalize(df.iloc[:, 0][0])

    for i in range(len(df)-1):
        aux = pd.json_normalize(df.iloc[:, 0][i+1])
        final = pd.concat([final, aux])

    return final

# **DF das médias dos últimos 4 anos para os 3 meses de previsão**

In [ ]:
def df_media4(dataframe):
    df = dataframe
    df = (df.iloc[:, 0]+df.iloc[:, 1]+df.iloc[:, 2]+df.iloc[:, 3])/4
    df = df.reset_index()
    return df

def calcula_medias(dataframe):
    df = dataframe
    total = len(df.columns)
    df1 = df.iloc[:, [total-12,total-24,total-36,total-48]]
    df2 = df.iloc[:, [total-11,total-23,total-35,total-47]]
    df3 = df.iloc[:, [total-10,total-22,total-34,total-46]]
    df1 = df_media4(df1)
    df2 = df_media4(df2)
    df3 = df_media4(df3)

    df_aux = df1
    df_aux.columns = ['CodProduto', 'media m1']
    df_aux['media m2'] = df2.iloc[:, 1]
    df_aux['media m3'] = df3.iloc[:, 1]
    df_aux.index = df_aux['CodProduto']
    df_aux = df_aux.iloc[:, 1:]
    return df_aux

In [ ]:
df_medias = calcula_medias(df_hist_nov_totMABC)

In [ ]:
df_medias

# **Distribuições Bimestral e Trimestral**

In [ ]:
def distribuicao(df_media_mes):
    df = df_media_mes
    comp = len(df.columns)

    df_soma = df.sum(axis = 1)
    df_soma = df_soma.replace(0.0, np.nan)

    df_resultado = df.div(df_soma, axis = 0)
    df_resultado = df_resultado.replace(np.nan, 1/comp)

    return df_resultado

In [ ]:
df_distr_tri = distribuicao(df_medias)
df_distr_tri.columns = ['mes1', 'mes2', 'mes3']
df_distr_bi =  distribuicao(df_medias.iloc[:, [0,1]])
df_distr_bi.columns = ['mes1', 'mes2']

In [ ]:
df_distr_tri

In [ ]:
df_distr_bi

# **Tratativa de dados**

In [ ]:
df_svmMABC.index = df_svmMABC['CodProduto']
df_svm2MABC.index = df_svm2MABC['CodProduto']
df_svm3MABC.index = df_svm3MABC['CodProduto']

In [ ]:
df = df_distr_tri

In [ ]:
df = df.multiply(df_svm3MABC['previsao quant 3M h1'], axis = 0)

In [ ]:
#df = df.round(0)

In [ ]:
df_trimestre_calculado = df

In [ ]:
df = df_distr_bi
df = df.multiply(df_svm2MABC['previsao quant 2M h1'], axis = 0)
#df = df.round(0)
df_bimestre_calculado = df

In [ ]:
df_bimestre_calculado

In [ ]:
df_trimestre_calculado

In [ ]:
df_mes_calculado = df_svmMABC['previsao quant M h1']
#df_mes_calculado = df_mes_calculado.round(0)

In [ ]:
df_mes_calculado

# **Valores Tratados**

In [ ]:
df_trimestre_calculado

In [ ]:
df_valores_tratados = df_trimestre_calculado.copy()
dftri = df_trimestre_calculado.copy()
dfbi = df_bimestre_calculado.copy()
dfmes = df_mes_calculado.copy()
df_valores_tratados['mes1'] = (3*dftri['mes1']+2*dfbi['mes1']+dfmes)/6
#df_valores_tratados = df_valores_tratados + 0.0000001

df_valores_tratados['mes2'] = (3*dftri['mes2']+2*dfbi['mes2'])/5
#df_valores_tratados = df_valores_tratados + 0.0000001
#df_valores_tratados = df_valores_tratados.round(0)

In [ ]:
df_valores_tratados

# **Valores Crus**

In [ ]:
df_valores_crus = df_valores_tratados.copy()
df_svm = df_svmMABC.copy()
df_svm2 = df_svm2MABC.copy()
df_svm3 = df_svm3MABC.copy()
df_valores_crus['mes1'] = df_svm['previsao quant M h1']

In [ ]:
df_svm['1M H1'] = df_svm['previsao quant M h1']
df_svm['2M H1'] = df_svm2['previsao quant 2M h1']
df_svm['3M H1'] = df_svm3['previsao quant 3M h1']

In [ ]:
df_svm = df_svm.iloc[:, [-3, -2, -1]]

In [ ]:
df_svm

In [ ]:
df_valores_crus['mes2'] = df_svm['2M H1']-df_svm['1M H1']

In [ ]:
df_valores_crus[df_valores_crus<0]=0.0

In [ ]:
df_valores_crus['mes3'] = df_svm['3M H1']-df_svm[['1M H1', '2M H1']].max(axis=1)

In [ ]:
df_valores_crus[df_valores_crus<0]=0.0

In [ ]:
df_valores_crus

# **Distribuições dos dados tratados e crus**

In [ ]:
df_dist_tratados = distribuicao(df_valores_tratados)

In [ ]:
df_dist_tratados['score'] = (df_dist_tratados['mes1']-0.333333).abs()+(df_dist_tratados['mes2']-0.333333).abs()+(df_dist_tratados['mes3']-0.333333).abs()

In [ ]:
df_dist_tratados

In [ ]:
df_dist_crus = distribuicao(df_valores_crus)
df_dist_crus['score'] = (df_dist_crus['mes1']-0.333333).abs()+(df_dist_crus['mes2']-0.333333).abs()+(df_dist_crus['mes3']-0.333333).abs()

In [ ]:
df_dist_crus

In [ ]:
df_novamente = df_dist_crus.score - df_dist_tratados.score

In [ ]:
df_novamente[df_novamente>=0]=0
df_novamente[df_novamente<0]=1

In [ ]:
df_novamente

In [ ]:
df_nova_distribuicao = df_dist_tratados.iloc[:, [0,1,2]]

In [ ]:
df_nova_distribuicao

In [ ]:
df_dist_media = (df_dist_tratados + df_dist_crus)/2

In [ ]:
df_dist_media = df_dist_media.iloc[:, [0,1,2]]

In [ ]:
df_dist_media

In [ ]:
df_nova_distribuicao[df_novamente==1]=df_dist_media[df_novamente==1]

In [ ]:
df_nova_distribuicao

In [ ]:
df_valores_tratados2 = df_nova_distribuicao.multiply(df_valores_tratados.sum(axis=1), axis=0)

In [ ]:
df_valores_tratados2

# **Estoque**

In [ ]:
df_estoque.index = df_estoque.CodProduto

In [ ]:
df_estoque = df_estoque.iloc[:, 1:-1]

In [ ]:
df_estoque

In [ ]:
df_estoque[df_estoque.index=='12701570']

In [ ]:
df_estoque_g = df_estoque.groupby(['CodProduto'])['QuantdadeItem'].sum()

In [ ]:
df_estoque_g = df_estoque_g.reset_index()

In [ ]:
df_estoque_g.index = df_estoque_g.CodProduto

In [ ]:
df_estoque_g = df_estoque_g.iloc[:, 1:]

In [ ]:
df_aux = df_valores_tratados2.copy()

In [ ]:
df_aux = df_aux.iloc[:, [0]]

In [ ]:
df_aux.columns = ['QuantdadeItem']

In [ ]:
df_aux

In [ ]:
df_aux.QuantdadeItem = df_aux.index

In [ ]:
# Intersecção e diferença
def inter_dif(lst1, lst2):
    intersection = set(lst1).intersection(lst2)
    difference1 = set(lst1).difference(set(lst2))
    difference2 = set(lst2).difference(set(lst1))
    print('total primeira: ', len(lst1))
    print('total segunda: ', len(lst2))
    print('intersecção: ', len(intersection))
    print('primeira - segunda: ', len(difference1))
    print('segunda - primeira', len(difference2))
    return list(intersection), list(difference1), list(difference2)

In [ ]:
list_estoque_g = list(df_estoque_g.index)
list_total = list(df_aux.index)
list_v_estoque = list(df_estoque_g.QuantdadeItem)

In [ ]:
inter = inter_dif(list_total, list_estoque_g)

In [ ]:
list_ntem = inter[1]

In [ ]:
mapping = dict(zip(list_estoque_g, list_v_estoque))
df_aux.QuantdadeItem = df_aux.QuantdadeItem.map(mapping)

In [ ]:
df_aux.QuantdadeItem = df_aux.QuantdadeItem.replace(list_ntem, 0.0)

In [ ]:
df_aux.loc[list_ntem]

In [ ]:
df_aux[df_aux.index=='12701570']

In [ ]:
df_estoque = df_aux

In [ ]:
df_estoque

In [ ]:
df_estoque[df_estoque.index=='030625']

# **Vendas Atualizadas**

In [ ]:
df_vendas_atualizadas = df_vendas_atualizadas_entrada.copy()

In [ ]:
df_vendas_atualizadas.columns = ['QuantdadeItem']

In [ ]:
df_vendas_atualizadas

In [ ]:
type(df_vendas_atualizadas_entrada.index[0])

In [ ]:
type(df_vendas_atualizadas.index[0])

In [ ]:
df_vendas_atualizadas.QuantdadeItem[0]

In [ ]:
df_aux = df_valores_tratados2.copy()
df_aux = df_aux.iloc[:, [0]]
df_aux.columns = ['QuantdadeItem']
df_aux.QuantdadeItem = df_aux.index

list_atual = list(df_vendas_atualizadas.index)
list_total = list(df_aux.index)
list_v_vendas_atualizadas = list(df_vendas_atualizadas.QuantdadeItem)

inter = inter_dif(list_total, list_atual)
list_inter = inter[0]
list_ntem = inter[1]
list_ntem_v = [0.0]*len(list_ntem)
list_zero = inter[2]

df_aux.QuantdadeItem = df_aux.index
mapping = dict(zip(list_atual, list_v_vendas_atualizadas))
df_aux.QuantdadeItem = df_aux.QuantdadeItem.map(mapping)
mapping = dict(zip(list_ntem, list_ntem_v))
df_aux.QuantdadeItem = df_aux.QuantdadeItem.map(mapping)

In [ ]:
df_vendas_atualizadas = df_aux

In [ ]:
df_vendas_atualizadas

# **Última sugestão**

In [ ]:
df_aux = df_ultima_sugestao_entrada.copy()

In [ ]:
df_aux = df_aux.rename(columns = {'NOVEMBRO.1':'mes1', 'DEZEMBRO.1':'mes2', 'JANEIRO.1':'mes3'})

In [ ]:
df_aux = df_aux[['mes2', 'mes3']]

In [ ]:
df_aux.columns = ['mes1', 'mes2']

In [ ]:
df_aux

In [ ]:
df_ultima_sugestao = df_vendas_atualizadas.copy()
df_ultima_sugestao.columns = ['mes1']
df_ultima_sugestao['mes2']= df_ultima_sugestao.index
df_ultima_sugestao['mes1']= df_ultima_sugestao.index

In [ ]:
list_total = list(df_ultima_sugestao.index)
list_parc = list(df_aux.index)

inter = inter_dif(list_total, list_parc)
list_inter = inter[0]
list_inter_m1 = list(df_aux.mes1)
list_inter_m2 = list(df_aux.mes2)
list_nan = inter[1]
list_nan_v = [0]*len(list_nan)

mapping = dict(zip(list_inter, list_inter_m1))
df_ultima_sugestao.mes1 = df_ultima_sugestao.mes1.map(mapping)
mapping = dict(zip(list_inter, list_inter_m2))
df_ultima_sugestao.mes2 = df_ultima_sugestao.mes2.map(mapping)
mapping = dict(zip(list_nan, list_nan_v))
df_ultima_sugestao.mes1 = df_ultima_sugestao.mes1.map(mapping)
df_ultima_sugestao.mes2 = df_ultima_sugestao.mes2.map(mapping)

In [ ]:
df_ultima_sugestao

# **Resultado**

In [ ]:
df_hist_12 = df_hist_nov_totMABC.iloc[:, -12:]

In [ ]:
df_hist_12

In [ ]:
df_aleatorio_cria = df_hist_nov_totMABC.copy()
df_aleatorio_cria = df_aleatorio_cria.iloc[:, :2]
df_aleatorio_cria.columns = ['Fixo1', 'Fixo2']
df_aleatorio_cria['Fixo1'] = np.random.randint(-1, 1, df_aleatorio_cria.shape[0])
df_aleatorio_cria['Fixo2'] = np.random.randint(-1, 1, df_aleatorio_cria.shape[0])

In [ ]:
df_aleatorio_cria

In [ ]:
# Salvando
#df_aleatorio.to_csv(r'df_aleatorio_20221214.csv')

In [ ]:
def calcula_max_hist(dataframe):
    df = dataframe
    df = df.iloc[:, -12:]
    df = df.max(axis=1)
    return(df)

In [ ]:
df_max_hist = calcula_max_hist(df_hist_nov_totMABC)

In [ ]:
df_linear = df_linear.iloc[:, -1:]

In [ ]:
df_linear.columns = ['mes1']

In [ ]:
c1 = df_linear[df_linear<0]
df_linear[df_linear<0]=0
df_linear['mes2'] = df_linear['mes1']+df_aleatorio['Fixo1']
c2 = df_linear[df_linear<0]
df_linear[df_linear<0]=0
df_linear['mes3'] = df_linear['mes2']+df_aleatorio['Fixo2']
c3 = df_linear[df_linear<0]
df_linear[df_linear<0]=0

In [ ]:
# usa o linear inicial para ver quantos negativos tem nele
#list_linear_negativo = list(df_linear[c1.notna().mes1].index)
#df_linear.loc[list_linear_negativo]
# Salvando
#df_linear.loc[list_linear_negativo].to_csv(r'df_lineares_errados.csv')

In [ ]:
df_linear

In [ ]:
df_max_hist[df_max_hist.index=='035503191']

In [ ]:
df_hibrido = df_valores_tratados2.copy()

In [ ]:
df_hibrido.mes1 = df_hibrido.index
df_hibrido.mes2 = df_hibrido.index
df_hibrido.mes3 = df_hibrido.index

In [ ]:
df_dif1 = df_max_hist - df_valores_tratados2.mes1
df_dif2 = df_max_hist - df_valores_tratados2.mes2
df_dif3 = df_max_hist - df_valores_tratados2.mes3

In [ ]:
list_dif1_svm = list(df_dif1[df_dif1>=0].index)
list_dif2_svm = list(df_dif2[df_dif2>=0].index)
list_dif3_svm = list(df_dif3[df_dif3>=0].index)

list_dif1_svm_v1 = list(df_valores_tratados2.loc[list_dif1_svm]['mes1'])
list_dif2_svm_v2 = list(df_valores_tratados2.loc[list_dif2_svm]['mes2'])
list_dif3_svm_v3 = list(df_valores_tratados2.loc[list_dif3_svm]['mes3'])

list_dif1_lin = list(df_dif1[df_dif1<0].index)
list_dif2_lin = list(df_dif2[df_dif2<0].index)
list_dif3_lin = list(df_dif3[df_dif3<0].index)

list_dif1_lin_v1 = list(df_linear.loc[list_dif1_lin]['mes1'])
list_dif2_lin_v2 = list(df_linear.loc[list_dif2_lin]['mes2'])
list_dif3_lin_v3 = list(df_linear.loc[list_dif3_lin]['mes3'])

In [ ]:
df_hist_mult = df_hist_12.prod(axis = 1)

In [ ]:
list_com_gap = list(df_hist_mult[df_hist_mult==0].index)
list_sem_gap = list(df_hist_mult[df_hist_mult!=0].index)

In [ ]:
teste = inter_dif(list_com_gap, list_sem_gap)

In [ ]:
list_com_gap_v1 = list(df_linear.loc[list_com_gap]['mes1'])
list_com_gap_v2 = list(df_linear.loc[list_com_gap]['mes2'])
list_com_gap_v3 = list(df_linear.loc[list_com_gap]['mes3'])

In [ ]:
mapping = dict(zip(list_com_gap, list_com_gap_v1))
df_hibrido.mes1 = df_hibrido.mes1.map(mapping)
mapping = dict(zip(list_com_gap, list_com_gap_v2))
df_hibrido.mes2 = df_hibrido.mes2.map(mapping)
mapping = dict(zip(list_com_gap, list_com_gap_v3))
df_hibrido.mes3 = df_hibrido.mes3.map(mapping)

In [ ]:
mapping = dict(zip(list_dif1_svm, list_dif1_svm_v1))
df_hibrido.mes1 = df_hibrido.mes1.map(mapping)
mapping = dict(zip(list_dif2_svm, list_dif2_svm_v2))
df_hibrido.mes2 = df_hibrido.mes2.map(mapping)
mapping = dict(zip(list_dif3_svm, list_dif3_svm_v3))
df_hibrido.mes3 = df_hibrido.mes3.map(mapping)

In [ ]:
mapping = dict(zip(list_dif1_lin, list_dif1_lin_v1))
df_hibrido.mes1 = df_hibrido.mes1.map(mapping)
mapping = dict(zip(list_dif2_lin, list_dif2_lin_v2))
df_hibrido.mes2 = df_hibrido.mes2.map(mapping)
mapping = dict(zip(list_dif3_lin, list_dif3_lin_v3))
df_hibrido.mes3 = df_hibrido.mes3.map(mapping)

In [ ]:
df_hibrido_sugestao = (df_hibrido + df_ultima_sugestao)/2

In [ ]:
list_media = list(df_hibrido_sugestao[df_hibrido_sugestao.mes1.notna()].index)
list_old = list(df_hibrido_sugestao[df_hibrido_sugestao.mes1.isna()].index)

In [ ]:
df_hibrido_sugestao.loc[list_old] = df_hibrido.loc[list_old]

In [ ]:
df_hibrido_sugestao['mes3'] = df_hibrido['mes3']

In [ ]:
df_hibrido_sugestao

In [ ]:
df_vendas_atualizadas

In [ ]:
# O cálculo a seguir usa:
# Atualização parcial : peso w1=1 para df_vendas_atualizadas e peso w2 = 2 para df_hibrido_sugestao
# Atualização total : peso w1=0 para df_vendas_atualizadas e peso w2 = 3 para df_hibrido_sugestao

In [ ]:
w1 = 1
w2 = 2
df_projecao_vendas = df_hibrido_sugestao.copy()
proj1 = (30-dia_mes)/dia_mes
proj2 = (30-dia_mes)/30
df_projecao_vendas['mes1'] = (df_vendas_atualizadas.QuantdadeItem*proj1*w1 + df_hibrido_sugestao.mes1*w2*proj2)/3 + df_vendas_atualizadas.QuantdadeItem

In [ ]:
df_projecao_vendas['mes2'] = ((df_projecao_vendas.mes1/df_hibrido_sugestao.mes1 - 1)/2 +1).multiply(df_hibrido_sugestao.mes2)

In [ ]:
df_projecao_vendas['mes3'] = ((df_projecao_vendas.mes1/df_hibrido_sugestao.mes1 - 1)/2 +1).multiply(df_hibrido_sugestao.mes3)

In [ ]:
df_projecao_vendas = df_projecao_vendas.replace(np.nan, 0.0)

In [ ]:
df_projecao_vendas = df_projecao_vendas + 0.000001

In [ ]:
df_projecao_vendas = df_projecao_vendas.round(0)

In [ ]:
df_projecao_vendas

In [ ]:
df_projecao_vendas.max(axis=1)

In [ ]:
# Refatorar a partir daqui e modificar df_max_hist adequadamente (com peso adequado)

In [ ]:
df_estoque_min = (df_max_hist + 2*df_projecao_vendas.max(axis=1))/3

In [ ]:
df_estoque_min = df_estoque_min + 0.000001
df_estoque_min = df_estoque_min.round(0)

In [ ]:
df_estoque_sugerido = df_estoque_min.copy()

In [ ]:
df_estoque_sugerido[df_estoque_sugerido>=10] = 1.2*df_estoque_sugerido[df_estoque_sugerido>=10]

In [ ]:
df_estoque_sugerido[df_estoque_sugerido<10] = df_estoque_sugerido[df_estoque_sugerido<10] + 2

In [ ]:
df_estoque_sugerido = df_estoque_sugerido + 0.000001
df_estoque_sugerido = df_estoque_sugerido.round(0)

In [ ]:
df_estoque_sugerido = df_estoque_sugerido.reset_index()

In [ ]:
df_estoque_sugerido.columns = ['CodProduto', 'QuantdadeItem']
df_estoque_sugerido.index = df_estoque_sugerido.CodProduto
df_estoque_sugerido = df_estoque_sugerido.iloc[:, 1:]

In [ ]:
df_sugestao_compra = df_estoque_sugerido - df_estoque

In [ ]:
df_sugestao_compra[df_sugestao_compra<0]=0

In [ ]:
df_final = df_ultima_sugestao_entradaABC.copy()
df_final = df_final.iloc[:, [0,1]]

In [ ]:
df_estoque

In [ ]:
df_hist_12

In [ ]:
df_final_final = pd.concat([df_final, df_hist_12, df_projecao_vendas, df_estoque, df_estoque_sugerido, df_sugestao_compra], axis=1)

In [ ]:
df_final_final.columns = ['Fornecedor', 'Curva',
                           'janeiro', 'fevereiro', 'março', 'abril', 'maio', 'junho', 'julho', 'agosto', 'setembro', 'outubro', 'novembro','dezembro',
                         'mes1', 'mes2', 'mes3', 'Estoque', 'Estoque_sugerido', 'Sugestao_Compra']

In [ ]:
df_final_final

In [ ]:
# Salvando
df_final_final.to_csv(r'df_sugestao_' + s_data + '.csv')

In [ ]:
#df_estoque.to_csv(r'df_produtos.csv')

In [ ]:
#df_produtos = df_estoque.copy()

In [ ]:
#df_produtos = df_produtos.reset_index()

In [ ]:
#df_produtos = df_produtos.iloc[:, 0]

In [ ]:
#df_produtos.to_csv(r'df_produtos.csv')

In [ ]:
df_final_final

In [ ]:
df_coberturas

In [ ]:
df_final_final.insert(loc = 1,
          column = 'cobertura',
          value = list(df_final_final.Fornecedor))

In [ ]:
lista_coberturas = list(df_coberturas.fornecedor)
lista_coberturas_v = list(df_coberturas.cobertura)
mapping = dict(zip(lista_coberturas, lista_coberturas_v))

df_final_final.cobertura = df_final_final.cobertura.map(mapping)

In [ ]:
df_final_final

In [ ]:
df_regressao = pd.DataFrame()
h1 = df_final_final.mes1
h2 = df_final_final.mes2
h3 = df_final_final.mes3
ac1 = h1
ac2 = h1 + h2
ac3 = ac2 + h3
df_regressao['b1'] = 30*ac1+60*ac2+90*ac3
df_regressao['b2'] = ac1+ac2+ac3
df_regressao['z1'] = df_regressao['b1']
df_regressao['z2'] = df_regressao['b2']-(df_regressao['z1'])/70
df_regressao['b0'] = 7*df_regressao['z2']/3
df_regressao['a0'] = (df_regressao['z1']-180*df_regressao['b0'])/12600
df_regressao['15'] = 15*df_regressao['a0']+ df_regressao['b0']
df_regressao['120'] = 120*df_regressao['a0']+ df_regressao['b0']
df_regressao['30'] = df_regressao['a0']*30+df_regressao['b0']
df_regressao['60'] = df_regressao['a0']*60+df_regressao['b0']
df_regressao['90'] = df_regressao['a0']*90+df_regressao['b0']
df_regressao['mes1'] = df_regressao['30']
df_regressao['mes2'] = df_regressao['60']-df_regressao['mes1']
df_regressao['mes3'] = df_regressao['90']-df_regressao['mes2']

In [ ]:
df_regressao

In [ ]:
df_regressao = df_regressao[['15','120','a0', 'b0']]

In [ ]:
df_regressao

In [ ]:
ac0 = df_regressao['15']
ac4 = df_regressao['120']

df_regressao['a1'] = (ac1-ac0)/15
df_regressao['b1'] = (30*ac0-15*ac1)/15
df_regressao['a2'] = (ac2-ac1)/30
df_regressao['b2'] = (60*ac1-30*ac2)/30
df_regressao['a3'] = (ac3-ac2)/30
df_regressao['b3'] = (90*ac2-60*ac3)/30
df_regressao['a4'] = (ac4-ac3)/30
df_regressao['b4'] = (120*ac3-90*ac4)/30

In [ ]:
df_regressao

In [ ]:
df_slug = df_final_final.copy()

In [ ]:
df_slug.loc[df_slug.cobertura<=15, 'novo1']=df_regressao.a0 * df_slug.cobertura + df_regressao.b0
df_slug.loc[(df_slug.cobertura<=30)&(df_slug.cobertura>15), 'novo1']=df_regressao.a1 * df_slug.cobertura + df_regressao.b1
df_slug.loc[(df_slug.cobertura<=60)&(df_slug.cobertura>30), 'novo1']=df_regressao.a2 * df_slug.cobertura + df_regressao.b2
df_slug.loc[(df_slug.cobertura<=90)&(df_slug.cobertura>60), 'novo1']=df_regressao.a3 * df_slug.cobertura + df_regressao.b3

df_slug.loc[df_slug.cobertura<=7.5, 'novo2']=df_regressao.a0 *2* df_slug.cobertura + df_regressao.b0
df_slug.loc[(df_slug.cobertura<=15)&(df_slug.cobertura>7.5), 'novo2']=df_regressao.a1 *2* df_slug.cobertura + df_regressao.b1
df_slug.loc[(df_slug.cobertura<=30)&(df_slug.cobertura>15), 'novo2']=df_regressao.a2 *2* df_slug.cobertura + df_regressao.b2
df_slug.loc[(df_slug.cobertura<=45)&(df_slug.cobertura>30), 'novo2']=df_regressao.a3 *2* df_slug.cobertura + df_regressao.b3

df_slug.loc[df_slug.cobertura<=5, 'novo3']=df_regressao.a0 *3* df_slug.cobertura + df_regressao.b0
df_slug.loc[(df_slug.cobertura<=10)&(df_slug.cobertura>5), 'novo3']=df_regressao.a1 *3* df_slug.cobertura + df_regressao.b1
df_slug.loc[(df_slug.cobertura<=20)&(df_slug.cobertura>10), 'novo3']=df_regressao.a2 *3* df_slug.cobertura + df_regressao.b2
df_slug.loc[(df_slug.cobertura<=30)&(df_slug.cobertura>20), 'novo3']=df_regressao.a3 *3* df_slug.cobertura + df_regressao.b3
df_slug.loc[(df_slug.cobertura<=40)&(df_slug.cobertura>30), 'novo3']=df_regressao.a4 *3* df_slug.cobertura + df_regressao.b4

In [ ]:
df_slug['h1']=df_slug['novo1']
df_slug['h2']=df_slug['novo2']-df_slug['novo1']
df_slug['h3']=df_slug['novo3']-df_slug['novo2']

In [ ]:
df_slug

In [ ]:
df_slug = df_slug.iloc[:, [0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,24,25,26]]

In [ ]:
df_slug

In [ ]:
df_pedidos = df_pedidos.reset_index()
df_pedidos.PRODUTO = df_pedidos.PRODUTO.str.strip()
df_abertos = df_pedidos.groupby(['PRODUTO'])['SALDO'].sum()
df_abertos = df_abertos.reset_index()

list_todos = list(df_final_final.index)
lista_abertos = list(df_abertos.PRODUTO)

xlok = inter_dif(list_todos, lista_abertos)
lista_abertos_int = xlok[0]
lista_n_abertos = xlok[1]

In [ ]:
df_abertos[df_abertos.PRODUTO.isin(lista_abertos_int)]

In [ ]:
df_slug['abertos']=df_slug.index

In [ ]:
lista_abertos_int = list(df_abertos[df_abertos.PRODUTO.isin(lista_abertos_int)].PRODUTO)
lista_abertos_int_v = list(df_abertos[df_abertos.PRODUTO.isin(lista_abertos_int)].SALDO)

In [ ]:
mapping = dict(zip(lista_abertos_int, lista_abertos_int_v))
df_slug.abertos = df_slug.abertos.map(mapping)
mapping = dict(zip(lista_n_abertos, [0]*len(lista_n_abertos)))
df_slug.abertos = df_slug.abertos.map(mapping)

In [ ]:
df_projecao_vendas2 = df_slug.iloc[:, [-4,-3,-2]]
df_emaberto = df_slug.iloc[:, 18:19]
df_emaberto.columns = ['QuantdadeItem']
df_estoque_soma = df_estoque.iloc[:,0:1]+df_emaberto
#(df_estoque_soma)
df_estoque_min = ((df_slug.cobertura*df_max_hist)/30 + 2*df_projecao_vendas2.max(axis=1))/3
df_estoque_min = df_estoque_min + 0.000001
df_estoque_min = df_estoque_min.round(0)

df_estoque_sugerido = df_estoque_min.copy()
df_estoque_sugerido[df_estoque_sugerido>=10] = 1.2*df_estoque_sugerido[df_estoque_sugerido>=10]
df_estoque_sugerido[df_estoque_sugerido<10] = df_estoque_sugerido[df_estoque_sugerido<10] + 2
df_estoque_sugerido = df_estoque_sugerido + 0.000001
df_estoque_sugerido = df_estoque_sugerido.round(0)
df_estoque_sugerido = df_estoque_sugerido.reset_index()
df_estoque_sugerido.columns = ['CodProduto', 'QuantdadeItem']
df_estoque_sugerido.index = df_estoque_sugerido.CodProduto
df_estoque_sugerido = df_estoque_sugerido.iloc[:, 1:]
df_sugestao_compra = df_estoque_sugerido - df_estoque_soma
df_sugestao_compra[df_sugestao_compra<0]=0

df_final_final_c = pd.concat([df_slug, df_estoque_soma, df_estoque_sugerido, df_sugestao_compra], axis=1)

df_final_final_c.columns = ['Fornecedor','cobertura', 'Curva',
                           'fevereiro', 'março', 'abril', 'maio', 'junho', 'julho', 'agosto', 'setembro', 'outubro', 'novembro','dezembro','janeiro',
                         'h1', 'h2', 'h3','abertos', 'Estoque', 'Estoque_sugerido', 'Sugestao_Compra']

df_final_final_c.h1 = df_final_final_c.h1 + 0.000001
df_final_final_c.h1 = df_final_final_c.h1.round(0)
df_final_final_c.h2 = df_final_final_c.h2 + 0.000001
df_final_final_c.h2 = df_final_final_c.h2.round(0)
df_final_final_c.h3 = df_final_final_c.h3 + 0.000001
df_final_final_c.h3 = df_final_final_c.h3.round(0)

df_final_final_c.loc[df_final_final_c.h1<0, ['h1']]=0
df_final_final_c.loc[df_final_final_c.h2<0, ['h2']]=0
df_final_final_c.loc[df_final_final_c.h3<0, ['h3']]=0

In [ ]:
# Salvando
df_final_final_c.to_csv(r'df_sugestao_'+ s_data + '_c.csv')